In [7]:
# Cell 1 - Imports
import os
from dotenv import load_dotenv
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.core import Settings

import sys
# setting path
# Aggiungi la cartella PARENT della cartella Parla_con_PG_TM
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# import from parent directory llm.py:
from Parla_con_PG_TM.llm import init_local_embed_model


load_dotenv()

True

In [8]:
# Cell 3 - Load existing vector store
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings, StorageContext, load_index_from_storage
PERSIST_DIR = project_root + "\Parla_con_PG_TM\chroma_db"

def load_vector_store():
    """Load the existing vector store from disk"""
    if not os.path.exists(PERSIST_DIR):
        raise ValueError(f"Storage directory '{PERSIST_DIR}' not found")
    Settings.embed_model = init_local_embed_model()
    print("Loading vector store...")
    chroma_client = chromadb.PersistentClient(path=PERSIST_DIR)
    collection = chroma_client.get_collection(name="ardania_lore")
    vector_store = ChromaVectorStore(chroma_collection=collection)
    index = VectorStoreIndex.from_vector_store(vector_store=vector_store)
    print("Vector store loaded successfully")
    return index

In [9]:

# Cell 4 - Query function
def query_similar_docs(index, query_text, top_k=3):
    """
    Retrieve top k most similar documents to the query
    """
    # Create query embedding and retrieve similar docs
    retriever = index.as_retriever(similarity_top_k=top_k)
    nodes = retriever.retrieve(query_text)
    
    print(f"\nTop {top_k} similar documents to '{query_text}':\n")
    for i, node in enumerate(nodes, 1):
        print(f"Document {i}:")
        print(f"Score: {node.score:.4f}")
        print(f"Content: {node.text}\n")
    
    return nodes

In [10]:
# Cell 5 - Execute query
if __name__ == "__main__":
    # Load index
    index = load_vector_store()
    
    # Query for similar documents
    query_text = "parlami di Helcaraxe e delle terre del nord"
    similar_docs = query_similar_docs(index, query_text,10)

Loading vector store...
Vector store loaded successfully

Top 10 similar documents to 'parlami di Helcaraxe e delle terre del nord':

Document 1:
Score: 0.6520
Content: onore dei Danu.Al tempio della dea vengono rinnovati i voti agli Dei e viene indetta una regata in onore della dea dei mari. Alcunidjaredin terranno uno spettacolo pirotecnico durante tale occasione, davanti a numerosi stranieri e molti altri cugini.Nelle prime schermaglie della citata guerra Eldor si trova coinvolta nei combattimenti, in aiuto dei nordici. Non avendoperò discusso di tali intenti con il popolo Djaredin, tutti i mezz’elfi vengono allontanati definitivamente da Nuran Kar.Altre ten

Document 2:
Score: 0.6452
Content: lainghiottirà per sempre.• Ettanien (“Il grande globo”), visibile specialmente di notte al riflesso delle 12 sorelle (di cui tratteremo più avanti).Il pianeta risulta essere assolutamente immobile nella volta superiore e si dice che sia un mondo abitato. Forse sonosoltanto leggende, ma nell’an

In [11]:
# print all the scores of similar documents
print("\nScores of similar documents:")
for i, doc in enumerate(similar_docs, 1):
    print(f"Document {i} Score: {doc.score:.4f}")
# End of script


Scores of similar documents:
Document 1 Score: 0.6520
Document 2 Score: 0.6452
Document 3 Score: 0.6447
Document 4 Score: 0.6433
Document 5 Score: 0.6417
Document 6 Score: 0.6361
Document 7 Score: 0.6349
Document 8 Score: 0.6344
Document 9 Score: 0.6335
Document 10 Score: 0.6328


In [15]:
similar_docs[0].metadata["headings"]

'Storia delle Razze di Ardania > Manoscritto - Storia degli Ultimi anni > Anno 275 > Orifoglia – Novità nelle terre umane'